<a href="https://colab.research.google.com/github/JariusF10-8/bt4222/blob/main/notebooks/models/feature-based_MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced model 1: Feature-based MLP

Purpose => Encode each article from its **text, entities and category**. Mean-pool the user's clicked-history article vectors into one **history vector**. Concatenate `[history vector ; candidate vector ; engineered matching/context features]` and score it with an **MLP** that outputs **one logit per candidate**.

```
   user's last ≤50 clicks                 candidate article
            │                                     │
   ┌────────▼────────┐                   ┌────────▼────────┐
   │ article encoder │  ← same weights → │ article encoder │   text + entities + category → 128-d
   └────────┬────────┘                   └────────┬────────┘
     mean-pool (masked)                           │
            │ h (128)                             │ c (128)
            └───────────────┬─────────────────────┘
          concat [ h ; c ; engineered features x ]
                            │
                MLP 128 → 64 (ReLU, dropout)
                            │
                        1 logit  ──sigmoid──►  estimated click probability
```

- **Loss:** `BCEWithLogitsLoss` on 1 click + 4 non-clicks from the same impression.
- **Ranking:** candidates are sorted by score **within each impression**, so no 0.5 threshold is needed.
- **Metrics:** AUC, MRR, nDCG@5, nDCG@10 per impression, then averaged. The same splits, sampler and metrics are used as in the other model notebooks.

## Roadmap

| Step | What happens | Data used |
|---|---|---|
| **0 · Setup** | Paths, starting hyperparameters, imports | none |
| **1 · Data** | Load splits; ranking metrics; negative sampling | train, validation |
| **2 · Features** | **Full feature list**; clean and transform the features | train (fit only) |
| **3 · Model** | Article encoder, mean-pooling, MLP head, training loop, quick checks | none |
| **4 · Build** | Train the model with the **starting hyperparameters** and all features | train (validation only for early stopping) |
| **5 · Diagnose** | Learning curves, train vs validation gap, user cohorts | validation |
| **6 · Tune features** | Permutation feature importance, then compare feature sets | validation |
| **7 · Tune hyperparameters** | Small search with the chosen feature set | validation |
| **8 · Finalise** | Lock the best features and hyperparameters, then **retrain on train + validation** | train + validation |
| **9 · Test once** | One evaluation of the final model on test | test |

> **Rule:** every choice in Steps 4–8 uses validation only. Test is read **once**, in Step 9.

---
# STEP 0 · Setup

> **Goal:** set the paths, the starting hyperparameters and the fixed training rules.
> **Output:** `STARTING_HYPERPARAMETERS`, `SEARCH_SPACE` and the run folder.

| Hyperparameter | Starting value | Searched in Step 7 |
|---|---|---|
| Text encoder | Frozen: TF-IDF → truncated SVD (fitted on train), article vectors cached | fixed |
| Combined article-vector dimension | 128 | 64, 128 |
| Scoring MLP | Two hidden layers 128 → 64, ReLU | [64], [128, 64] |
| Dropout | 0.1 | 0, 0.1, 0.3 |
| Learning rate | 0.001 | 0.0003, 0.001 |
| Optimizer / weight decay | AdamW / 0.00001 | 0, 0.00001, 0.0001 |
| Maximum history length | 50 articles, with padding masks | 20, 50 |
| Batch size | 128 sampled groups = 640 user–candidate pairs (same as `Candidate_Attention_MLP`) | fixed |
| Training limit | 15 epochs; early-stopping patience 3 on validation nDCG@10 | fixed |

Locally `STORAGE_ROOT=None` finds this repository. In Colab the shared Drive folder is mounted. Outputs go to `outputs/models/feature_mlp/<RUN_NAME>/`.

In [1]:
# ══ STEP 0 · Setup ═══════════════════════════════════════════════════════
from pathlib import Path
import sys, json, time, random, hashlib, itertools, importlib.util, subprocess
from datetime import datetime, timezone

# ── Paths ────────────────────────────────────────────────────────────────
STORAGE_ROOT = None               # None: shared Drive folder in Colab, this repository locally
PROCESSED_VERSION = 'v2'
FEATURE_VERSION = 'unified_v1'
RUN_NAME = 'feature_mlp_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')

# ── Starting hyperparameters (project plan) ──────────────────────────────
STARTING_HYPERPARAMETERS = {
    'dim': 128,              # combined article-vector dimension
    'hidden': [128, 64],     # scoring MLP hidden layers (ReLU)
    'dropout': 0.1,
    'learning_rate': 1e-3,   # AdamW
    'weight_decay': 1e-5,    # AdamW
    'max_history': 50,       # most recent clicks kept; padding is masked
}

# ── Search space for Step 7 ──────────────────────────────────────────────
SEARCH_SPACE = {
    'dim': [64, 128],
    'hidden': [[64], [128, 64]],
    'dropout': [0.0, 0.1, 0.3],
    'learning_rate': [3e-4, 1e-3],
    'weight_decay': [0.0, 1e-5, 1e-4],
    'max_history': [20, 50],
}
N_TRIALS = 6                      # starting setting + its 64-d variant + 4 random grid points

# ── Fixed training rules ─────────────────────────────────────────────────
BATCH_GROUPS = 128                # 128 groups x (1 click + 4 non-clicks) = 640 pairs per step
NEGATIVES_PER_CLICK = 4
MAX_EPOCHS, PATIENCE, MIN_DELTA = 15, 3, 1e-4
TEXT_COMPONENTS = 256             # TF-IDF -> truncated SVD size
CATEGORY_DIM = 8                  # size of the category and subcategory embeddings
SEED, TIE_SEED = 42, 2026         # TIE_SEED: same label-free tie order as the other notebooks

# ── Environment ──────────────────────────────────────────────────────────
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
except ImportError:
    IN_COLAB = False
if STORAGE_ROOT is None:
    STORAGE_ROOT = ('/content/drive/MyDrive/BT4222 Group Project' if IN_COLAB else
                    next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/data_loading.py').is_file()), None))
    if STORAGE_ROOT is None:
        raise FileNotFoundError('Open this notebook inside the bt4222 repository, or set STORAGE_ROOT.')
STORAGE_ROOT = Path(STORAGE_ROOT).expanduser().resolve()
PROCESSED_DIR = STORAGE_ROOT / 'data/processed' / PROCESSED_VERSION
FEATURE_DIR = next((p for p in [STORAGE_ROOT / 'outputs/features' / FEATURE_VERSION, STORAGE_ROOT / 'data' / FEATURE_VERSION]
                    if (p / 'manifest.json').is_file()), None)
RUN_DIR = STORAGE_ROOT / 'outputs/models/feature_mlp' / RUN_NAME
if not (PROCESSED_DIR / 'manifest.json').is_file() or FEATURE_DIR is None:
    raise FileNotFoundError(f'Need {PROCESSED_DIR} and the {FEATURE_VERSION} feature bundle; this notebook builds neither.')

required = {'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy', 'sklearn': 'scikit-learn',
            'matplotlib': 'matplotlib', 'torch': 'torch'}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing and IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
elif missing:
    raise ImportError(f'Missing {missing} in this kernel ({sys.executable}): pip install {" ".join(missing)}')

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, normalize
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import display


def require(condition, message):
    if not condition:
        raise ValueError(message)


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, default=str) + '\n', encoding='utf-8')


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)


FEATURE_MANIFEST = json.loads((FEATURE_DIR / 'manifest.json').read_text())
require(FEATURE_MANIFEST['source_manifest_sha256'] == sha256_file(PROCESSED_DIR / 'manifest.json'),
        'Feature bundle and preprocessing manifest do not match.')
require(not RUN_DIR.exists(), 'Choose a new RUN_NAME; existing runs are preserved.')
RUN_DIR.mkdir(parents=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
seed_everything(SEED)
print('Preprocessed input:', PROCESSED_DIR, '\nFeature bundle:', FEATURE_DIR, '\nRun folder:', RUN_DIR)
print('Device:', DEVICE, '| PyTorch:', torch.__version__)

Mounted at /content/drive
Preprocessed input: /content/drive/.shortcut-targets-by-id/1ZMBqrHA3wOM8h1DuBW04gypymjJiPmk1/BT4222 Group Project/data/processed/v2 
Feature bundle: /content/drive/.shortcut-targets-by-id/1ZMBqrHA3wOM8h1DuBW04gypymjJiPmk1/BT4222 Group Project/outputs/features/unified_v1 
Run folder: /content/drive/.shortcut-targets-by-id/1ZMBqrHA3wOM8h1DuBW04gypymjJiPmk1/BT4222 Group Project/outputs/models/feature_mlp/feature_mlp_20261108_043738
Device: cpu | PyTorch: 2.14.1+cpu


---
# STEP 1 · Data, metrics and negative sampling

> **Goal:** load train and validation, define the ranking metrics, and define how training samples are drawn.
> **Data used:** train, validation. **Test is not loaded until Step 9.**

**1.1 Loading.** Each impression keeps its 50 most recent history clicks, right-aligned, with row 0 used as padding.

**1.2 Metrics.** MIND-style AUC, MRR, nDCG@5 and nDCG@10 are computed per impression and then averaged. Only impressions with at least one click and one non-click count. Ties use a fixed, label-independent random order.

**1.3 Cohorts.** A *cold user* has no training impression. An impression is marked *any cold item* when a candidate was never a training candidate. Impressions are also grouped by history length; `history_0` means an empty history.

In [2]:
# ══ STEP 1.1–1.3 · Load data, metrics, cohorts ═══════════════════════════
KEYS = ['event_key', 'candidate_position', 'news_id']
METRICS = ['AUC', 'MRR', 'nDCG@5', 'nDCG@10']
HISTORY_BUCKETS = [(0, 0, 'history_0'), (1, 9, 'history_1_9'), (10, 29, 'history_10_29'),
                   (30, 49, 'history_30_49'), (50, 10**9, 'history_50_plus')]


def read_partition(name):
    """Events, candidates and a (events x 50) matrix of history article rows (0 = padding)."""
    folder = PROCESSED_DIR / name
    events = pd.read_csv(folder / 'events.csv', usecols=['event_key', 'user_id', 'timestamp', 'history_ids'],
                         dtype='string', keep_default_na=False)
    events['timestamp'] = pd.to_datetime(events['timestamp'])
    history = events.pop('history_ids').map(json.loads)
    candidates = pd.read_csv(folder / 'candidates.csv', usecols=KEYS + ['label'],
                             dtype={'event_key': 'string', 'news_id': 'string', 'candidate_position': 'int32',
                                    'label': 'int8'}, keep_default_na=False)
    news_ids = pd.read_csv(FEATURE_DIR / name / 'news_index.csv', dtype={'news_id': 'string'})['news_id']
    articles = pd.Index(news_ids)
    event_row = pd.Index(events['event_key']).get_indexer(candidates['event_key'])
    article_row = articles.get_indexer(candidates['news_id'])
    lengths = history.map(len).to_numpy('int64')
    flat = articles.get_indexer([n for h in history for n in h])
    require((event_row >= 0).all() and (article_row >= 0).all() and (flat >= 0).all(), f'{name}: unaligned inputs')
    # Right-aligned matrix of the most recent 50 clicks; article rows start at 1 because 0 is padding.
    cap = max(SEARCH_SPACE['max_history'] + [STARTING_HYPERPARAMETERS['max_history']])
    keep = np.minimum(lengths, cap)
    first = np.r_[0, np.cumsum(lengths)][:-1] + lengths - keep
    within = np.arange(keep.sum()) - np.repeat(np.cumsum(keep) - keep, keep)
    matrix = np.zeros((len(events), cap), dtype='int64')
    matrix[np.repeat(np.arange(len(events)), keep), np.repeat(cap - keep, keep) + within] = \
        flat[np.repeat(first, keep) + within] + 1
    return dict(name=name, events=events, candidates=candidates, news_ids=news_ids, event_row=event_row,
                article_row=article_row, history=matrix, history_length=lengths,
                y=candidates['label'].to_numpy('float32'), tie=np.random.default_rng(TIE_SEED).random(len(candidates)))


def ranking_metrics(part, scores, per_impression=False):
    """MIND-style macro metrics over impressions with a click and a non-click; ties use a fixed random order."""
    data = pd.DataFrame({'g': part['event_row'], 'y': part['y'].astype('int64'), 's': np.asarray(scores, 'float64'),
                         'tie': part['tie']}).sort_values(['g', 's', 'tie'], ascending=[True, False, False], kind='stable')
    data['rank'] = data.groupby('g', sort=False).cumcount() + 1
    data['avg_rank'] = data.groupby('g', sort=False)['s'].rank(method='average')
    n, pos = data.groupby('g', sort=False)['y'].size(), data.groupby('g', sort=False)['y'].sum()
    data['rr'] = data['y'] / data['rank']
    data['gain'] = data['y'] / np.log2(data['rank'] + 1)
    data['pos_rank'] = data['y'] * data['avg_rank']
    sums = data.groupby('g', sort=False)[['rr', 'pos_rank']].sum()
    table = pd.DataFrame({'AUC': (sums['pos_rank'] - pos * (pos + 1) / 2) / (pos * (n - pos)), 'MRR': sums['rr'] / pos})
    ideal = np.r_[0., np.cumsum(1 / np.log2(np.arange(2, 12)))]
    for k in (5, 10):
        dcg = data.loc[data['rank'] <= k].groupby('g', sort=False)['gain'].sum().reindex(n.index, fill_value=0.)
        table[f'nDCG@{k}'] = dcg / ideal[np.minimum(pos.to_numpy(int), k)]
    table = table.loc[(pos > 0) & (pos < n)]
    return table if per_impression else {'impressions': len(table), **table.mean().to_dict()}


def attach_cohorts(part, seen_users, seen_articles):
    """Flag cold users, impressions with an unseen candidate, and history-length buckets."""
    cold_user = ~part['events']['user_id'].isin(seen_users).to_numpy()
    any_cold_item = np.zeros(len(part['events']), dtype=bool)
    np.logical_or.at(any_cold_item, part['event_row'], ~part['candidates']['news_id'].isin(seen_articles).to_numpy())
    bucket = np.empty(len(part['events']), dtype=object)
    for low, high, name in HISTORY_BUCKETS:
        bucket[(part['history_length'] >= low) & (part['history_length'] <= high)] = name
    part['cohorts'] = pd.DataFrame({'cold_user': cold_user, 'any_cold_item': any_cold_item, 'history': bucket})


def cohort_report(part, scores):
    table = ranking_metrics(part, scores, per_impression=True)
    flags = part['cohorts'].iloc[table.index.to_numpy()].reset_index(drop=True)
    table = table.reset_index(drop=True)
    masks = {'all': np.ones(len(table), bool), 'warm_user': ~flags['cold_user'], 'cold_user': flags['cold_user'],
             'all_items_warm': ~flags['any_cold_item'], 'any_cold_item': flags['any_cold_item'],
             **{name: flags['history'] == name for _, _, name in HISTORY_BUCKETS}}
    return pd.DataFrame([{'cohort': name, 'impressions': int(mask.sum()), **table.loc[mask, METRICS].mean().to_dict()}
                         for name, mask in masks.items()])


train, validation = read_partition('train'), read_partition('validation')
require(train['events']['timestamp'].max() < validation['events']['timestamp'].min(), 'Train/validation overlap')
TRAIN_USERS, TRAIN_ARTICLES = set(train['events']['user_id']), set(train['candidates']['news_id'])
for part in (train, validation):
    attach_cohorts(part, TRAIN_USERS, TRAIN_ARTICLES)
print(f"Train:      {len(train['events']):,} impressions, {len(train['y']):,} candidates, {int(train['y'].sum()):,} clicks")
print(f"Validation: {len(validation['events']):,} impressions, {len(validation['y']):,} candidates")
print(f"Validation cold users: {validation['cohorts']['cold_user'].mean():.1%} of impressions; "
      f"empty histories: {(validation['history_length'] == 0).mean():.1%}")

Train:      95,071 impressions, 3,389,342 candidates, 141,558 clicks
Validation: 61,894 impressions, 2,454,102 candidates
Validation cold users: 24.5% of impressions; empty histories: 2.4%


### 1.4 Negative sampling (training samples)

This is the repository's `src/sampling.py` rule, copied unchanged, so every model notebook trains on the same samples for a given seed.

- Each training **click** is grouped with **4 non-clicks from the same impression**. Non-clicks are drawn with replacement only when fewer than 4 exist.
- Negatives are **redrawn every epoch** with seed `seed + epoch`.
- Validation and test always score **complete impressions** and are never sampled. The one exception is the sampled validation loss in Step 5, which is only a diagnostic.

In [3]:
# ══ STEP 1.4 · Negative sampling ═════════════════════════════════════════
def sample_negatives(candidates, k=4, seed=0):
    """One group per click: the click (slot 0) plus k non-clicks from the same impression (src/sampling.py)."""
    rng = np.random.default_rng(seed)
    order = ['event_key', 'candidate_position']
    positives = candidates.loc[candidates['label'].eq(1)].sort_values(order, kind='stable')
    negatives = candidates.loc[candidates['label'].eq(0)].sort_values(order, kind='stable').reset_index(drop=True)
    counts = negatives.groupby('event_key', sort=False).size()
    starts = counts.cumsum() - counts
    positives = positives.loc[positives['event_key'].map(counts).fillna(0).gt(0)].reset_index(drop=True)
    n = positives['event_key'].map(counts).to_numpy('int64')[:, None]
    offsets = rng.integers(0, n, size=(len(positives), k))
    while True:  # Redraw rows with repeats wherever enough distinct negatives exist.
        ordered = np.sort(offsets, axis=1)
        repeated = (ordered[:, 1:] == ordered[:, :-1]).any(axis=1) & (n[:, 0] >= k)
        if not repeated.any():
            break
        offsets[repeated] = rng.integers(0, n[repeated], size=(int(repeated.sum()), k))
    group_ids = np.arange(len(positives))
    picks = (positives['event_key'].map(starts).to_numpy('int64')[:, None] + offsets).ravel()
    sampled = pd.concat([positives.assign(group_id=group_ids, slot=0),
                         negatives.iloc[picks].assign(group_id=np.repeat(group_ids, k),
                                                      slot=np.tile(np.arange(1, k + 1), len(positives)))])
    return sampled.sort_values(['group_id', 'slot'], kind='stable').reset_index(drop=True)


def training_groups(part, epoch, seed):
    """(groups, 5) matrix of candidate rows for one epoch; column 0 is the click."""
    sampled = sample_negatives(part['candidates'], NEGATIVES_PER_CLICK, seed + epoch)
    key = pd.MultiIndex.from_frame(part['candidates'][['event_key', 'candidate_position']])
    rows = key.get_indexer(pd.MultiIndex.from_frame(sampled[['event_key', 'candidate_position']]))
    return rows.reshape(-1, NEGATIVES_PER_CLICK + 1)


def epoch_batches(parts, epoch, seed):
    """Shuffled list of (partition, groups) batches for one epoch; `parts` is [train] or [train, validation]."""
    rng = np.random.default_rng(seed + 100_000 + epoch)
    batches = []
    for part in parts:
        groups = training_groups(part, epoch, seed)
        groups = groups[rng.permutation(len(groups))]
        batches += [(part, groups[s:s + BATCH_GROUPS]) for s in range(0, len(groups), BATCH_GROUPS)]
    return [batches[i] for i in rng.permutation(len(batches))]


# Fixed sampled validation groups: used only for the validation-loss curve in Step 5.
VAL_GROUPS = training_groups(validation, 0, SEED)
print(f"{len(training_groups(train, 1, SEED)):,} training groups per epoch "
      f"({NEGATIVES_PER_CLICK + 1} candidates each); {len(VAL_GROUPS):,} sampled validation groups")

141,558 training groups per epoch (5 candidates each); 94,786 sampled validation groups


---
# STEP 2 · Features

> **Goal:** list every model input, remove useless engineered columns, and fit the transforms.
> **Data used:** **train only** for every fitted rule (SVD, scalers, category maps, cleaning). Validation and test are only *transformed*.

## 2.1 Complete feature list

### A. Learned article representation (from the article encoder)

Every article, whether a candidate or a history click, goes through the **same** encoder:

| # | Input | Size | How it is built |
|---|---|---|---|
| 1 | **Text vector** | 256 + 1 missing flag | Title + abstract TF-IDF (bundle) → truncated SVD fitted on training articles → L2-normalised |
| 2 | **Entity vector** | 100 + 1 missing flag | Mean of the article's Wikidata entity embeddings (bundle `uniform` variant) → L2-normalised |
| 3 | **Category embedding** | 8 | Learned embedding of `category_id`; categories unseen in training map to index 0 |
| 4 | **Subcategory embedding** | 8 | Learned embedding of `subcategory_id`; same rule |

`[1;2;3;4]` (374 numbers) → `Linear → LayerNorm → Dropout` → **128-d article vector**.

| Derived vector | Meaning |
|---|---|
| **Candidate vector `c`** | The candidate's 128-d article vector |
| **History vector `h`** | The masked **mean** of the article vectors of the user's last ≤ 50 clicks. An empty history uses a learned vector. |

### B. Engineered features (go straight into the MLP head, next to `h` and `c`)

127 columns from the unified feature bundle (`data/unified_v1`), in 7 groups:

| Group | Level | # | Columns | Meaning |
|---|---|---|---|---|
| **Article text & category flags** | article | 7 | `metadata_missing`, `title_missing`, `abstract_missing`, `title_length`, `abstract_length`, `category_unknown`, `subcategory_backoff` | Text availability and length; whether the category is rare or unknown |
| **Article entity statistics** | article | 34 | `title_entities_missing`, `abstract_entities_missing`, `num_title_entities`, `num_abstract_entities`, `num_embedded_entities`, `entity_conf_mean`, `entity_conf_min`, `entity_conf_missing`, `entity_vector_missing_{uniform, prominence, title_only, confidence}`, `has_entity_at_title_start`, `title_entity_absent`, `title_entity_first_position`, `abstract_entity_absent`, `abstract_entity_first_position`, `title_start_type_{person, organisation, place, other, none}`, `num_{person, organisation, place}_entities`, `{person, organisation, place}_entities_missing`, `{person, organisation, place}_entities_without_embedding`, `{person, organisation, place}_embedding_missing` | How many entities the article has, of which type, where they appear, and how confident the annotations are |
| **Impression context** | impression | 4 | `history_length`, `num_candidates`, `hour_sin`, `hour_cos` | How much history the user has, slate size, time of day |
| **History profile** | impression | 31 | `history_known_category_count`, `history_category_missing`, `history_category_share_00 … _17`, `history_subcategory_missing`, `history_{person, organisation, place}_missing`, `history_{person, organisation, place}_embedding_missing`, `history_entity_vector_missing_{uniform, prominence, title_only, confidence}` | The user's category mix and which history signals are missing |
| **Candidate–history category match** | candidate | 3 | `candidate_history_category_share`, `candidate_history_subcategory_count`, `candidate_history_subcategory_share` | How often the user clicked this candidate's category or subcategory before |
| **Candidate–history entity match** | candidate | 10 | `entity_similarity_{uniform, prominence, title_only, confidence}`, `{person, organisation, place}_entity_overlap`, `{person, organisation, place}_embedding_similarity` | How close the candidate's entities are to the user's history entities |
| **Time × category interactions** | candidate | 38 | `hour_sin__x__candidate_category_00 … _17`, `hour_sin__x__history_category_share`, `hour_cos__x__candidate_category_00 … _17`, `hour_cos__x__history_category_share` | Whether some categories, or category matches, matter more at certain hours |

### C. Not used (by design)

| Columns | Reason |
|---|---|
| `candidate_category_00 … _17` (one-hot) | Category is already a learned embedding (A3) |
| `day_of_week`, `is_weekend`, `weekday_sin`, `weekday_cos` | Weekday inputs are disabled by the bundle (`weekday_model_inputs_enabled = false`) |
| `hour_of_day`, `fractional_hour` | Not bundle model columns; `hour_sin` / `hour_cos` encode the hour |
| Historical CTR / popularity (`art_prior_*`, `art_*_1h/6h/24h`, `subcat_prior_ctr`, `user_prior_*`, `user_article_*`) | Left out, as in the other model notebooks |
| `user_id`, `news_id` | No ID embeddings, so unseen users and articles are scored from content (cold start) |

## 2.2 Cleaning rule (train only)

Within each level, in the bundle's column order, a column is dropped when:
1. **Near-constant:** its most common value covers more than 99.9% of training rows, or
2. **Duplicate:** |Pearson r| > 0.95 with a column already kept.

Whether a feature actually *helps the model* is decided later, from validation, in Step 6.

In [ ]:
# ══ STEP 2.1–2.2 · Feature catalogue and training-only cleaning ═══════════
LEVELS = {'article': 'news_features.csv', 'event': 'event_features.csv', 'candidate': 'candidate_features.csv.gz'}
WEEKDAY = {'day_of_week', 'is_weekend', 'weekday_sin', 'weekday_cos'}
MAX_DOMINANT, MAX_CORRELATION, CLEAN_SAMPLE_ROWS = 0.999, 0.95, 300_000
ARTICLE_TEXT_FLAGS = {'metadata_missing', 'title_missing', 'abstract_missing', 'title_length', 'abstract_length',
                      'category_unknown', 'subcategory_backoff'}


def feature_group(name):
    """The engineered-feature group (Section 2.1 B) a column belongs to."""
    if '__x__' in name:
        return 'time x category'
    if name in ('history_length', 'num_candidates', 'hour_sin', 'hour_cos'):
        return 'impression context'
    if name.startswith('history_'):
        return 'history profile'
    if name.startswith('candidate_history_'):
        return 'category match'
    if name.startswith('entity_similarity') or name.endswith(('_entity_overlap', '_embedding_similarity')):
        return 'entity match'
    if name in ARTICLE_TEXT_FLAGS:
        return 'article text & category flags'
    return 'article entity statistics'


def feature_pool():
    """Engineered columns offered to the model, per level, plus the declared columns that are left out."""
    declared = FEATURE_MANIFEST['numeric_model_columns']
    pool, used = {}, set()
    for level, file in LEVELS.items():
        present = set(pd.read_csv(FEATURE_DIR / 'train' / file, nrows=0).columns)
        pool[level] = [c for c in declared if c in present and c not in WEEKDAY
                       and not c.startswith('candidate_category_')]
        used |= set(pool[level])
    left_out = pd.DataFrame([{'feature': c, 'reason': 'category is embedded instead' if c.startswith('candidate_category_')
                              else 'weekday disabled' if c in WEEKDAY else 'historical CTR/popularity'}
                             for c in declared if c not in used])
    return pool, left_out


def load_tables(part, columns):
    """Feature tables aligned with the partition's articles, impressions and candidate rows."""
    folder = FEATURE_DIR / part['name']
    a = pd.read_csv(folder / LEVELS['article'], usecols=['news_id', *columns['article']], dtype={'news_id': 'string'})
    require(a['news_id'].equals(part['news_ids']), 'news_features.csv must follow news_index.csv')
    e = pd.read_csv(folder / LEVELS['event'], usecols=['event_key', *columns['event']], dtype={'event_key': 'string'})
    e = e.set_index('event_key').loc[part['events']['event_key']].reset_index(drop=True)
    c = pd.read_csv(folder / LEVELS['candidate'], usecols=KEYS + columns['candidate'],
                    dtype={'event_key': 'string', 'news_id': 'string', 'candidate_position': 'int32'})
    require(c[KEYS].equals(part['candidates'][KEYS]), 'Candidate features differ from the source rows')
    tables = {'article': a[columns['article']], 'event': e[columns['event']], 'candidate': c[columns['candidate']]}
    tables = {level: t.astype('float32') for level, t in tables.items()}
    require(all(np.isfinite(t.to_numpy()).all() for t in tables.values()), 'Non-finite features')
    return tables


def clean_features(tables, pool):
    """Drop near-constant and duplicate columns, using training rows only."""
    rng = np.random.default_rng(SEED)
    kept, records = {}, []
    for level in LEVELS:
        frame = tables[level][pool[level]]
        if len(frame) > CLEAN_SAMPLE_ROWS:
            frame = frame.iloc[np.sort(rng.choice(len(frame), CLEAN_SAMPLE_ROWS, replace=False))]
        with np.errstate(invalid='ignore', divide='ignore'):
            corr = np.nan_to_num(np.corrcoef(frame.to_numpy('float64'), rowvar=False))
        kept[level], kept_idx = [], []
        for j, column in enumerate(pool[level]):
            top_share = frame[column].value_counts(normalize=True).iloc[0]
            twin = max(kept_idx, key=lambda i: abs(corr[j, i]), default=None)
            r = abs(corr[j, twin]) if twin is not None else 0.
            if top_share > MAX_DOMINANT:
                decision = 'drop: near-constant'
            elif r > MAX_CORRELATION:
                decision = f'drop: duplicate of {pool[level][twin]}'
            else:
                decision = 'keep'
                kept[level].append(column)
                kept_idx.append(j)
            records.append({'level': level, 'group': feature_group(column), 'feature': column,
                            'top_value_share': top_share, 'max_abs_corr_with_kept': r, 'decision': decision})
    return kept, pd.DataFrame(records)


tick = time.time()
POOL, LEFT_OUT = feature_pool()
TRAIN_TABLES = load_tables(train, POOL)
CLEAN, FEATURE_TABLE = clean_features(TRAIN_TABLES, POOL)
ALL_FEATURES = [c for level in LEVELS for c in CLEAN[level]]
GROUP_OF = {c: feature_group(c) for c in ALL_FEATURES}
ENGINEERED_GROUPS = list(dict.fromkeys(GROUP_OF.values()))
FEATURE_TABLE.to_csv(RUN_DIR / 'feature_table.csv', index=False)

print(f'{len(FEATURE_TABLE)} engineered columns offered, {len(ALL_FEATURES)} kept after cleaning ({time.time() - tick:.0f}s)')
print(f'{len(LEFT_OUT)} declared bundle columns not used:', LEFT_OUT.groupby('reason').size().to_dict())
display(pd.crosstab(FEATURE_TABLE['group'], FEATURE_TABLE['decision'].str.split(':').str[0], margins=True))
print('Dropped columns:')
display(FEATURE_TABLE.loc[FEATURE_TABLE['decision'] != 'keep', ['level', 'group', 'feature', 'decision']]
        .reset_index(drop=True))

## 2.3 Transforms (fitted on train, applied to every split)

| Input | Transform |
|---|---|
| Text | TF-IDF → `TruncatedSVD(256)` fitted on training articles → L2-normalise, plus a missing flag |
| Entities | L2-normalise, plus a missing flag |
| Category / subcategory | Map to embedding indices; unseen in training → 0 |
| Engineered features | `log1p` when a column is nonnegative, exceeds 1 and is strongly skewed (training skew > 2); then `StandardScaler` |

Validation, test and the final train + validation refit all use these **same** training-fitted transforms, so the features stay consistent across steps.

In [ ]:
# ══ STEP 2.3 · Fit transforms on train, apply to validation ══════════════
def unit_with_flag(matrix):
    """L2-normalised rows plus a missing flag for all-zero rows, with padding row 0 on top."""
    flag = (np.linalg.norm(matrix, axis=1) == 0)[:, None]
    block = np.column_stack([normalize(matrix), flag]).astype('float32')
    return np.vstack([np.zeros((1, block.shape[1]), 'float32'), block])


class FeaturePipeline:
    """Training-fitted text SVD, category maps, log1p choices and scalers."""

    def fit(self, part, tables):
        folder = FEATURE_DIR / 'train'
        news = pd.read_csv(folder / 'news_features.csv', usecols=['metadata_missing', 'category_id', 'subcategory_id'])
        text = sparse.load_npz(folder / 'tfidf_title_abstract.npz').tocsr()
        self.svd = TruncatedSVD(TEXT_COMPONENTS, random_state=SEED).fit(text[~news['metadata_missing'].astype(bool).to_numpy()])
        self.category_maps = [{v: j + 1 for j, v in enumerate(sorted(set(news[c]) - {0}))}
                              for c in ('category_id', 'subcategory_id')]
        tables = {level: t[CLEAN[level]] for level, t in tables.items()}
        self.log_columns = {level: [c for c in t if t[c].min() >= 0 and t[c].max() > 1 and t[c].skew() > 2]
                            for level, t in tables.items()}
        self.scalers = {level: StandardScaler().fit(self._log(level, t)) for level, t in tables.items()}
        return self.transform(part, tables)

    def _log(self, level, table):
        return table.assign(**{c: np.log1p(table[c]) for c in self.log_columns[level]}).to_numpy('float32')

    def transform(self, part, tables=None):
        """Adds part['articles'] (text+entity inputs), part['categories'] and part['side'] (engineered features)."""
        tables = load_tables(part, CLEAN) if tables is None else tables
        folder = FEATURE_DIR / part['name']
        text = self.svd.transform(sparse.load_npz(folder / 'tfidf_title_abstract.npz')).astype('float32')
        with np.load(folder / 'article_entity_vectors.npz') as z:
            require(np.array_equal(z['news_id'], part['news_ids'].to_numpy(str)), 'Entity vector order differs')
            entity = z['uniform'].astype('float32')
        part['articles'] = np.column_stack([unit_with_flag(text), unit_with_flag(entity)])
        cats = pd.read_csv(folder / 'news_features.csv', usecols=['category_id', 'subcategory_id'])
        codes = np.column_stack([cats[c].map(m).fillna(0).to_numpy('int64')
                                 for c, m in zip(('category_id', 'subcategory_id'), self.category_maps)])
        part['categories'] = np.vstack([np.zeros((1, 2), 'int64'), codes])
        part['side'] = {level: self.scalers[level].transform(self._log(level, t)).astype('float32')
                        for level, t in tables.items()}
        part['side']['article'] = np.vstack([np.zeros((1, part['side']['article'].shape[1]), 'float32'),
                                             part['side']['article']])
        return part


PIPELINE = FeaturePipeline()
PIPELINE.fit(train, TRAIN_TABLES)
del TRAIN_TABLES
PIPELINE.transform(validation)
ARTICLE_DIM = train['articles'].shape[1]          # 257 text + 101 entity
TEXT_COLUMNS = np.arange(TEXT_COMPONENTS + 1)
ENTITY_COLUMNS = np.arange(TEXT_COMPONENTS + 1, ARTICLE_DIM)
CATEGORY_SIZES = [len(m) + 1 for m in PIPELINE.category_maps]
print(f'Text SVD explains {PIPELINE.svd.explained_variance_ratio_.sum():.1%} of training TF-IDF variance')
print(f'Article encoder input: {ARTICLE_DIM} text/entity + {2 * CATEGORY_DIM} category dims | '
      f'engineered features: {len(ALL_FEATURES)}')
print('log1p columns:', {k: v for k, v in PIPELINE.log_columns.items() if v})

---
# STEP 3 · Model and training loop

> **Goal:** define the network, the training function and the prediction function, then run quick correctness checks.

| Part | Definition |
|---|---|
| **Article encoder** | `Linear(374 → dim) → LayerNorm → Dropout`, shared by history and candidate articles |
| **History vector** | Masked mean of the last `max_history` history article vectors; empty history → learned vector |
| **MLP head** | `[h ; c ; x]` → hidden layers (ReLU + dropout) → **1 logit** |
| **Loss / optimiser** | `BCEWithLogitsLoss`; AdamW (weight decay on weight matrices and embeddings only); gradients clipped at norm 5 |
| **Early stopping** | Keep the epoch with the best **validation nDCG@10**; stop after 3 epochs without improvement (max 15) |

`train_model(...)` is used in two ways:
- **Tuning runs (Steps 4–7):** train on `train`, early-stop on validation.
- **Final run (Step 8):** train on `train + validation` for a fixed number of epochs. There is no validation left, so the epoch count comes from tuning.

In [ ]:
# ══ STEP 3.1 · Model ═════════════════════════════════════════════════════
class FeatureMLP(nn.Module):
    """Article encoder -> mean-pooled history vector -> MLP over [history; candidate; engineered features]."""

    def __init__(self, config, features):
        super().__init__()
        self.config, self.features = dict(config), list(features)
        # Positions of the chosen engineered features inside each level's table.
        chosen = set(features)
        self.select = {level: np.array([i for i, c in enumerate(CLEAN[level]) if c in chosen], dtype=int)
                       for level in LEVELS}
        dim, dropout = config['dim'], config['dropout']
        self.categories = nn.ModuleList([nn.Embedding(n, CATEGORY_DIM, padding_idx=0) for n in CATEGORY_SIZES])
        self.encoder = nn.Sequential(nn.Linear(ARTICLE_DIM + 2 * CATEGORY_DIM, dim), nn.LayerNorm(dim), nn.Dropout(dropout))
        self.empty_history = nn.Parameter(torch.zeros(dim))
        layers, width = [], 2 * dim + len(self.features)
        for hidden in config['hidden']:
            layers += [nn.Linear(width, hidden), nn.ReLU(), nn.Dropout(dropout)]
            width = hidden
        self.head = nn.Sequential(*layers, nn.Linear(width, 1))

    def encode(self, features, categories):
        """Article inputs -> dim-d article vectors."""
        return self.encoder(torch.cat([features, *[emb(categories[:, j]) for j, emb in enumerate(self.categories)]], 1))

    def pool(self, history_vectors, mask):
        """Masked mean over (G, L, d) history vectors; empty histories get the learned vector."""
        count = mask.sum(1, keepdim=True)
        mean = (history_vectors * mask[..., None]).sum(1) / count.clamp(min=1)
        return torch.where(count > 0, mean, self.empty_history)

    def score(self, candidates, user, side):
        """candidates (G, C, d), user (G, d), side (G, C, s) -> logits (G, C)."""
        return self.head(torch.cat([user[:, None].expand_as(candidates), candidates, side], -1)).squeeze(-1)

    def forward(self, features, categories, history, mask, candidates, side):
        vectors = self.encode(features, categories)
        return self.score(vectors[candidates], self.pool(vectors[history], mask), side)


def side_features(part, rows, select):
    """Chosen engineered features for candidate rows: [article | impression | candidate] columns."""
    return np.concatenate([part['side']['article'][np.ix_(part['article_row'][rows] + 1, select['article'])],
                           part['side']['event'][np.ix_(part['event_row'][rows], select['event'])],
                           part['side']['candidate'][np.ix_(rows, select['candidate'])]], axis=1)


def make_batch(part, groups, model):
    """Tensors for a (G, C) block of candidate rows sharing one impression per row of `groups`."""
    history = part['history'][part['event_row'][groups[:, 0]]][:, -model.config['max_history']:]
    candidates = part['article_row'][groups] + 1
    unique, inverse = np.unique(np.concatenate([history.ravel(), candidates.ravel()]), return_inverse=True)
    t = lambda a: torch.as_tensor(a, device=DEVICE)
    return (t(part['articles'][unique]), t(part['categories'][unique]), t(inverse[:history.size].reshape(history.shape)),
            t(history > 0), t(inverse[history.size:].reshape(candidates.shape)),
            t(side_features(part, groups.ravel(), model.select).reshape(*groups.shape, -1)))


@torch.inference_mode()
def predict(model, part, block=65_536):
    """One logit per candidate row of `part` (complete impressions). Every article is encoded once."""
    model.eval()
    vectors = model.encode(torch.as_tensor(part['articles'], device=DEVICE), torch.as_tensor(part['categories'], device=DEVICE))
    history = torch.as_tensor(part['history'][:, -model.config['max_history']:], device=DEVICE)
    users = torch.cat([model.pool(vectors[h], h > 0) for h in history.split(4_096)])
    scores = []
    for start in range(0, len(part['y']), block):
        r = np.arange(start, min(start + block, len(part['y'])))
        cand = vectors[torch.as_tensor(part['article_row'][r] + 1, device=DEVICE)][:, None]
        side = torch.as_tensor(side_features(part, r, model.select)[:, None], device=DEVICE)
        user = users[torch.as_tensor(part['event_row'][r], device=DEVICE)]
        scores.append(model.score(cand, user, side)[:, 0].cpu().numpy())
    return np.concatenate(scores)


@torch.inference_mode()
def sampled_bce(model, part, groups, block=1_024):
    """BCE on fixed 1-click + 4-non-click groups, comparable with the training loss."""
    model.eval()
    total = 0.
    for s in range(0, len(groups), block):
        g = groups[s:s + block]
        logits = model(*make_batch(part, g, model))
        total += F.binary_cross_entropy_with_logits(logits, torch.as_tensor(part['y'][g], device=DEVICE),
                                                    reduction='sum').item()
    return total / groups.size

In [ ]:
# ══ STEP 3.2 · Training function ═════════════════════════════════════════
def train_model(name, config, features, parts=None, epochs=None, seed=SEED):
    """Train one FeatureMLP.

    epochs=None -> tuning run: train on `train`, up to MAX_EPOCHS, early stopping on validation nDCG@10,
                   and keep the best epoch's weights.
    epochs=k    -> final run: train on `parts` (e.g. train + validation) for exactly k epochs.
    """
    parts = parts or [train]
    tuning = epochs is None
    seed_everything(seed)
    model = FeatureMLP(config, features).to(DEVICE)
    decay = [p for p in model.parameters() if p.ndim > 1]
    other = [p for p in model.parameters() if p.ndim <= 1]
    optimizer = torch.optim.AdamW([{'params': decay, 'weight_decay': config['weight_decay']},
                                   {'params': other, 'weight_decay': 0.}], lr=config['learning_rate'])
    loss_fn = nn.BCEWithLogitsLoss()
    best_score, best_state, best_epoch, stale, curve = -np.inf, None, 0, 0, []

    for epoch in range(1, (MAX_EPOCHS if tuning else epochs) + 1):
        tick = time.time()
        model.train()
        total, count = 0., 0
        for part, batch in epoch_batches(parts, epoch, seed):
            optimizer.zero_grad()
            loss = loss_fn(model(*make_batch(part, batch, model)), torch.as_tensor(part['y'][batch], device=DEVICE))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.)
            optimizer.step()
            total, count = total + loss.item() * batch.size, count + batch.size
        row = {'epoch': epoch, 'train_bce': total / count}
        message = f"{name} | epoch {epoch:02d} | train BCE {row['train_bce']:.4f}"

        if tuning:  # Validation check: sampled loss + full-impression ranking metrics.
            row['val_bce'] = sampled_bce(model, validation, VAL_GROUPS)
            row.update(ranking_metrics(validation, predict(model, validation)))
            message += f" | val BCE {row['val_bce']:.4f} | val AUC {row['AUC']:.4f} | val nDCG@10 {row['nDCG@10']:.4f}"
            if row['nDCG@10'] > best_score + MIN_DELTA:
                best_score, best_epoch, stale = row['nDCG@10'], epoch, 0
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                stale += 1
        curve.append(row)
        print(message + f' | {time.time() - tick:.0f}s', flush=True)
        if tuning and stale >= PATIENCE:
            break

    if tuning:
        model.load_state_dict(best_state)
    else:
        best_epoch = epochs
    model.name, model.best_epoch, model.curve = name, best_epoch, pd.DataFrame(curve)
    model.curve.to_csv(RUN_DIR / f'{name}_curve.csv', index=False)
    torch.save({'config': model.config, 'features': model.features, 'state': model.state_dict()}, RUN_DIR / f'{name}.pt')
    return model.eval()


def summarise(model):
    """One results row: settings + validation metrics at the kept epoch."""
    best = model.curve.loc[model.curve['epoch'] == model.best_epoch].iloc[0]
    return {'run': model.name, **model.config, 'n_features': len(model.features), 'best_epoch': model.best_epoch,
            'parameters': sum(p.numel() for p in model.parameters()), **{m: best[m] for m in METRICS}}

### 3.3 Quick checks (run before any training)

- the metric on a hand-computed example;
- sampled groups contain one click and four non-clicks from the same impression;
- mean pooling ignores padding and history order, and an empty history gets the learned vector;
- the training path and the fast prediction path give the same scores.

In [ ]:
# ══ STEP 3.3 · Quick checks ══════════════════════════════════════════════
def run_checks():
    toy = {'y': np.array([1, 0, 1], 'float32'), 'event_row': np.zeros(3, int), 'tie': np.array([.2, .3, .1])}
    m = ranking_metrics(toy, np.array([3., 2., 1.]))
    np.testing.assert_allclose([m['MRR'], m['AUC']], [(1 + 1 / 3) / 2, 0.5])

    groups = training_groups(train, 1, SEED)
    require((train['y'][groups[:, 0]] == 1).all() and (train['y'][groups[:, 1:]] == 0).all(), 'Bad sampled groups')
    require((train['event_row'][groups] == train['event_row'][groups[:, :1]]).all(), 'A group mixes impressions')

    seed_everything(SEED)
    model = FeatureMLP(STARTING_HYPERPARAMETERS, ALL_FEATURES).to(DEVICE).eval()
    dim = STARTING_HYPERPARAMETERS['dim']
    with torch.no_grad():
        model.empty_history.normal_()
        v = torch.randn(6, dim, device=DEVICE)
        pooled = model.pool(v[torch.tensor([[0, 3, 4, 5], [0, 0, 0, 0]], device=DEVICE)],
                            torch.tensor([[False, True, True, True], [False] * 4], device=DEVICE))
        torch.testing.assert_close(pooled[0], v[3:6].mean(0))
        torch.testing.assert_close(pooled[1], model.empty_history)
        reordered = model.pool(v[torch.tensor([[5, 0, 3, 4]], device=DEVICE)],
                               torch.tensor([[True, False, True, True]], device=DEVICE))
        torch.testing.assert_close(reordered[0], pooled[0])

    rows = np.random.default_rng(1).choice(len(validation['y']), 2_000, replace=False)
    fast = predict(model, validation)[rows]
    with torch.inference_mode():
        slow = model(*make_batch(validation, rows[:, None], model))[:, 0].cpu().numpy()
    np.testing.assert_allclose(fast, slow, rtol=1e-4, atol=1e-5)
    print('Checks passed: metrics, sampled groups, mean pooling and empty history, training/prediction agreement.')


run_checks()

---
# STEP 4 · Build the model on the training set

> **Goal:** train the feature-based MLP with the **starting hyperparameters** and **all cleaned engineered features**.
> **Data used:** fitted on **train**. Validation is used only to choose the best epoch (early stopping).
> **Output:** `BASELINE`, the reference model that Steps 5–7 diagnose and improve.

In [ ]:
# ══ STEP 4 · Build the baseline model ════════════════════════════════════
BASELINE = train_model('step4_baseline', STARTING_HYPERPARAMETERS, ALL_FEATURES)
display(pd.DataFrame([summarise(BASELINE)]).round(4))

---
# STEP 5 · Diagnose training performance with validation

> **Goal:** check whether the baseline is under-fitting, over-fitting or about right, and where it is weak.
> **Data used:** train (complete impressions) and validation.

**How to read the results**

| What you see | Likely meaning | What to try in Step 7 |
|---|---|---|
| Train BCE keeps falling, **validation BCE rises**, and validation nDCG@10 peaks early | Over-fitting | More dropout or weight decay; smaller `dim` |
| Train and validation are both poor and still improving at epoch 15 | Under-fitting | Higher learning rate; larger `dim` / MLP |
| Train metrics ≫ validation metrics | Over-fitting or a train/validation shift (validation is a later time period) | Regularise; check cohorts |
| Weak `cold_user` / `history_0` cohorts | The history vector carries little information there | Expected; engineered context features should help |

Both BCE curves use **sampled 1:4 groups**, so they are comparable. The ranking metrics use **complete impressions**.

In [ ]:
# ══ STEP 5.1 · Learning curves ═══════════════════════════════════════════
curve = BASELINE.curve
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(curve['epoch'], curve['train_bce'], marker='o', label='train')
axes[0].plot(curve['epoch'], curve['val_bce'], marker='o', label='validation')
axes[0].set(xlabel='Epoch', ylabel='BCE (sampled 1 click + 4 non-clicks)', title='Loss')
axes[1].plot(curve['epoch'], curve['nDCG@10'], marker='o', color='#2a78d6')
axes[1].set(xlabel='Epoch', ylabel='Validation nDCG@10', title='Validation ranking (complete impressions)')
for ax in axes:
    ax.axvline(BASELINE.best_epoch, color='#52514e', linestyle=':', label='kept epoch')
    ax.legend(frameon=False)
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(RUN_DIR / 'step5_learning_curve.png', dpi=160)
plt.show()

In [ ]:
# ══ STEP 5.2 · Train vs validation (complete impressions) ════════════════
BASELINE_VAL_SCORES = predict(BASELINE, validation)
gap = pd.DataFrame([
    {'data': 'train', **ranking_metrics(train, predict(BASELINE, train))},
    {'data': 'validation', **ranking_metrics(validation, BASELINE_VAL_SCORES)},
    {'data': 'validation, random order (floor)', **ranking_metrics(validation, np.zeros(len(validation['y'])))},
]).set_index('data')
gap.loc['gap (train - validation)', METRICS] = gap.loc['train', METRICS] - gap.loc['validation', METRICS]
gap.to_csv(RUN_DIR / 'step5_train_vs_validation.csv')
display(gap.round(4))

In [ ]:
# ══ STEP 5.3 · Where is the model weak? (validation cohorts) ═════════════
cohorts = cohort_report(validation, BASELINE_VAL_SCORES).set_index('cohort')
cohorts['random order nDCG@10'] = cohort_report(validation, np.zeros(len(validation['y']))).set_index('cohort')['nDCG@10']
cohorts.to_csv(RUN_DIR / 'step5_validation_cohorts.csv')
display(cohorts.round(4))

fig, ax = plt.subplots(figsize=(8, 4.4))
y = np.arange(len(cohorts))
ax.barh(y, cohorts['nDCG@10'], height=0.55, color='#2a78d6', label='Feature-based MLP', zorder=2)
ax.vlines(cohorts['random order nDCG@10'], y - 0.38, y + 0.38, color='#52514e', linewidth=2, label='Random order', zorder=3)
ax.set_yticks(y, cohorts.index)
ax.invert_yaxis()
ax.set_xlabel('Validation nDCG@10')
ax.set_title('Baseline: validation ranking by cohort', loc='left')
ax.grid(axis='x', alpha=0.3)
ax.legend(frameon=False, loc='lower right')
fig.tight_layout()
fig.savefig(RUN_DIR / 'step5_validation_cohorts.png', dpi=160)
plt.show()

---
# STEP 6 · Tune the features

> **Goal:** find out which inputs the model relies on, and pick the feature set with the best validation nDCG@10.
> **Data used:** validation.

### 6.1 Permutation feature importance

For each group, its values are **shuffled across rows** of validation (at the group's own level: articles, impressions or candidate rows). Validation is then re-scored with the **same trained baseline**. The importance is the **drop** in nDCG@10 and AUC.
- **Large drop:** the model depends on that group.
- **About 0 or negative:** the group adds nothing the model uses.

The learned inputs (text, entity, category, and the whole user history) are tested in the same way, for context.

In [ ]:
# ══ STEP 6.1 · Permutation importance on validation ══════════════════════
IMPORTANCE_REPEATS = 2


def importance_targets(part, model):
    """(group, kind, array, first row, columns) for every input group the model uses."""
    targets = [('text vector', 'learned', part['articles'], 1, TEXT_COLUMNS),
               ('entity vector', 'learned', part['articles'], 1, ENTITY_COLUMNS),
               ('category embeddings', 'learned', part['categories'], 1, np.arange(2)),
               ('user history (whole)', 'learned', part['history'], 0, np.arange(part['history'].shape[1]))]
    for group in ENGINEERED_GROUPS:
        for level in LEVELS:
            cols = np.array([i for i in model.select[level] if GROUP_OF[CLEAN[level][i]] == group], dtype=int)
            if len(cols):
                targets.append((group, 'engineered', part['side'][level], 1 if level == 'article' else 0, cols))
    return targets


def permutation_importance(model, part):
    base = ranking_metrics(part, predict(model, part))
    rows = []
    for group, kind, array, start, cols in importance_targets(part, model):
        original = array[start:, cols].copy()
        drops = []
        for r in range(IMPORTANCE_REPEATS):
            array[start:, cols] = original[np.random.default_rng(SEED + r).permutation(len(original))]
            shuffled = ranking_metrics(part, predict(model, part))
            drops.append({f'drop_{m}': base[m] - shuffled[m] for m in ('AUC', 'nDCG@10')})
        array[start:, cols] = original  # restore
        rows.append({'group': group, 'kind': kind, 'n_columns': len(cols), **pd.DataFrame(drops).mean().to_dict()})
        print(f"{group:32s} nDCG@10 drop {rows[-1]['drop_nDCG@10']:+.4f}", flush=True)
    return pd.DataFrame(rows).sort_values('drop_nDCG@10', ascending=False).reset_index(drop=True)


tick = time.time()
IMPORTANCE = permutation_importance(BASELINE, validation)
IMPORTANCE.to_csv(RUN_DIR / 'step6_permutation_importance.csv', index=False)
print(f'({time.time() - tick:.0f}s)')
display(IMPORTANCE.round(4))

fig, ax = plt.subplots(figsize=(8, 0.4 * len(IMPORTANCE) + 1))
colors = IMPORTANCE['kind'].map({'learned': '#9aa6b2', 'engineered': '#2a78d6'})
ax.barh(IMPORTANCE['group'], IMPORTANCE['drop_nDCG@10'], color=colors)
ax.axvline(0, color='#52514e', linewidth=1)
ax.invert_yaxis()
ax.set_xlabel('Drop in validation nDCG@10 when shuffled (higher = more important)')
ax.set_title('Permutation importance (blue = engineered, grey = learned inputs)', loc='left')
ax.grid(axis='x', alpha=0.3)
fig.tight_layout()
fig.savefig(RUN_DIR / 'step6_permutation_importance.png', dpi=160)
plt.show()

### 6.2 Compare feature sets

Each candidate set is retrained with the **starting hyperparameters**, so only the features change:

| Feature set | Engineered features in the MLP head |
|---|---|
| `all features` | All cleaned engineered features (this is the Step 4 baseline, reused) |
| `drop unhelpful groups` | Removes every engineered group whose importance drop in nDCG@10 is ≤ 0 |
| `no engineered features` | None: the head sees only `[h ; c]` |

The set with the highest validation nDCG@10 becomes `SELECTED_FEATURES`. A set that duplicates another is skipped.

In [ ]:
# ══ STEP 6.2 · Train and compare feature sets ════════════════════════════
unhelpful = set(IMPORTANCE.loc[(IMPORTANCE['kind'] == 'engineered') & (IMPORTANCE['drop_nDCG@10'] <= 0), 'group'])
print('Unhelpful engineered groups:', sorted(unhelpful) or 'none')
FEATURE_SETS = {
    'all features': ALL_FEATURES,
    'drop unhelpful groups': [c for c in ALL_FEATURES if GROUP_OF[c] not in unhelpful],
    'no engineered features': [],
}

FEATURE_RUNS, seen = {'all features': BASELINE}, {tuple(ALL_FEATURES)}
for set_name, features in FEATURE_SETS.items():
    if tuple(features) in seen:
        continue
    seen.add(tuple(features))
    FEATURE_RUNS[set_name] = train_model('step6_' + set_name.replace(' ', '_'), STARTING_HYPERPARAMETERS, features)

feature_results = pd.DataFrame([{'feature set': n, **summarise(m)} for n, m in FEATURE_RUNS.items()])
feature_results.to_csv(RUN_DIR / 'step6_feature_sets.csv', index=False)
display(feature_results[['feature set', 'n_features', 'best_epoch', *METRICS]].sort_values('nDCG@10', ascending=False).round(4))

SELECTED_SET = max(FEATURE_RUNS, key=lambda n: summarise(FEATURE_RUNS[n])['nDCG@10'])
SELECTED_FEATURES = FEATURE_SETS[SELECTED_SET]
print(f'Selected feature set: {SELECTED_SET} ({len(SELECTED_FEATURES)} engineered features)')

---
# STEP 7 · Tune the hyperparameters

> **Goal:** with `SELECTED_FEATURES` fixed, search the hyperparameters in `SEARCH_SPACE` (Step 0).
> **Data used:** train (fit), validation (early stopping and selection).

`N_TRIALS` settings are tried:
- trial 0 is the **starting setting**, already trained in Step 6 and reused;
- trial 1 is the starting setting with `dim = 64`;
- the remaining trials are drawn at random (fixed seed) from the full grid.

Every trial uses the same seed and the same negative samples. The best validation nDCG@10 wins; ties go to the model with fewer parameters.

In [ ]:
# ══ STEP 7 · Hyperparameter search ═══════════════════════════════════════
def search_configs(n):
    keys = list(SEARCH_SPACE)
    grid = [dict(zip(keys, values)) for values in itertools.product(*SEARCH_SPACE.values())]
    first = [STARTING_HYPERPARAMETERS, {**STARTING_HYPERPARAMETERS, 'dim': 64}]
    rest = [c for c in grid if c not in first]
    return (first + [rest[j] for j in np.random.default_rng(SEED).permutation(len(rest))])[:n]


TRIALS = []
for i, config in enumerate(search_configs(N_TRIALS)):
    if config == STARTING_HYPERPARAMETERS:
        TRIALS.append(FEATURE_RUNS[SELECTED_SET])      # already trained in Step 6
    else:
        TRIALS.append(train_model(f'step7_trial_{i:02d}', config, SELECTED_FEATURES))

search = pd.DataFrame([summarise(m) for m in TRIALS])
search.to_csv(RUN_DIR / 'step7_hyperparameter_search.csv', index=False)
display(search[['run', *STARTING_HYPERPARAMETERS, 'best_epoch', 'parameters', *METRICS]]
        .sort_values('nDCG@10', ascending=False).round(4))

BEST = max(TRIALS, key=lambda m: (summarise(m)['nDCG@10'], -summarise(m)['parameters']))
print('Best tuned model:', BEST.name, BEST.config, '| best epoch', BEST.best_epoch)

---
# STEP 8 · Finalise: lock the choices and retrain on train + validation

> **Goal:** fix every choice, then fit the final model on **more data**.
> **Data used:** train + validation (both used for fitting). Nothing is left for early stopping, so the model trains for the **best epoch count found in Step 7**.

Locked choices:
- **Features:** `SELECTED_FEATURES` (Step 6)
- **Hyperparameters:** `BEST.config` (Step 7)
- **Epochs:** `BEST.best_epoch` (Step 7)
- **Transforms:** the training-fitted `PIPELINE` (Step 2), unchanged

The same sampler draws 1 click + 4 non-clicks from **both** training and validation impressions.

In [ ]:
# ══ STEP 8 · Lock choices and retrain on train + validation ══════════════
FINAL_CONFIG, FINAL_EPOCHS = dict(BEST.config), BEST.best_epoch
LOCKED = {'features': SELECTED_FEATURES, 'feature_set': SELECTED_SET, 'hyperparameters': FINAL_CONFIG,
          'epochs': FINAL_EPOCHS, 'selected_from': BEST.name, 'validation_metrics_of_selected': summarise(BEST),
          'final_fit_partitions': ['train', 'validation'], 'test_used_for_selection': False}
write_json(RUN_DIR / 'step8_locked_choices.json', LOCKED)
print('Locked:', {k: LOCKED[k] for k in ('feature_set', 'hyperparameters', 'epochs')})

FINAL_MODEL = train_model('step8_final_train_plus_validation', FINAL_CONFIG, SELECTED_FEATURES,
                          parts=[train, validation], epochs=FINAL_EPOCHS)

---
# STEP 9 · Evaluate on the test set (once)

> **Goal:** report the final number.
> **Data used:** test, read here for the first time.

Set `EVALUATE_TEST = True` in this cell **only after Step 8 is done**. The cell refuses to run twice for the same run folder. **Report the test row**; do not change anything after seeing it.

- Test uses the training-fitted transforms (Step 2).
- Cold users and cold items are now defined relative to **train + validation**, the data the final model learned from.
- The train and validation rows are **in-sample** for the final model. They are shown for reference only.

In [ ]:
# ══ STEP 9 · Test once ═══════════════════════════════════════════════════
EVALUATE_TEST = False    # set to True once Step 8 is finished

if not EVALUATE_TEST:
    print('Test evaluation skipped. Set EVALUATE_TEST = True in this cell when every choice is locked.')
else:
    require(not (RUN_DIR / 'test_results.csv').exists(), 'This run already evaluated test.')
    test = read_partition('test')
    require(validation['events']['timestamp'].max() < test['events']['timestamp'].min(), 'Validation/test overlap')
    attach_cohorts(test, TRAIN_USERS | set(validation['events']['user_id']),
                   TRAIN_ARTICLES | set(validation['candidates']['news_id']))
    PIPELINE.transform(test)

    test_scores = predict(FINAL_MODEL, test)
    test_results = pd.DataFrame([
        {'data': 'TEST (final result)', **ranking_metrics(test, test_scores)},
        {'data': 'test, random order (floor)', **ranking_metrics(test, np.zeros(len(test['y'])))},
        {'data': 'train (in-sample)', **ranking_metrics(train, predict(FINAL_MODEL, train))},
        {'data': 'validation (in-sample)', **ranking_metrics(validation, predict(FINAL_MODEL, validation))},
    ]).set_index('data')
    test_results.to_csv(RUN_DIR / 'test_results.csv')
    test_cohorts = cohort_report(test, test_scores)
    test_cohorts.to_csv(RUN_DIR / 'test_cohorts.csv', index=False)
    display(test_results.round(4), test_cohorts.round(4))

---
# Findings

Run `feature_mlp_20261108_032209`, on a local CPU with seed 42, at about 13 s per epoch. Every selection used validation only. Test was scored once, in Step 9.

## Summary

| Item | Result |
|---|---|
| Final features | **All 90 cleaned engineered features** (127 offered, 37 dropped as near-constant or duplicate) |
| Final hyperparameters | **The starting setting:** dim 128, MLP 128 → 64, dropout 0.1, lr 0.001, AdamW weight decay 0.00001, history 50. (The search table rounds the weight decay to `0.0000`.) |
| Final training | Train + validation, **3 epochs** (the best epoch from tuning) |
| **Test result** | **AUC 0.6561 · MRR 0.3103 · nDCG@5 0.3417 · nDCG@10 0.4051** |
| Random-order floor on test | AUC 0.5000 · MRR 0.2182 · nDCG@5 0.2229 · nDCG@10 0.2865 |

## Step 5: diagnosis of the baseline

- **Learning curve.** Validation BCE was lowest at epoch 2 (0.4755) and stayed flat after that, while training BCE kept falling (0.4545 → 0.4311). Validation nDCG@10 peaked at epoch 3 (0.3921). Mild over-fitting therefore begins after about 3 epochs. Early stopping stopped the run at epoch 6 and kept epoch 3.
- **Train vs validation** (complete impressions): nDCG@10 was 0.5037 vs 0.3921 (gap 0.112), and AUC 0.7481 vs 0.6566 (gap 0.092). Part of this gap is the time shift rather than memorisation. Validation is a later period, and 96% of validation impressions contain at least one candidate never seen in training (`any_cold_item`).
- **Cohorts.** The model beats random order in every cohort.
  - **Cold users** score almost the same as warm users (nDCG@10 0.3912 vs 0.3924). Content features handle unseen users, because no user IDs are used.
  - **Empty histories** (`history_0`) are the weakest group: 0.2970 vs a random 0.2648.
  - **The gain over random grows with history length**, from +0.032 for `history_0` to about +0.13 for 30 or more clicks. So the history vector carries real signal.

## Step 6: feature tuning

**Permutation importance** (drop in validation nDCG@10 when the group is shuffled):

| Rank | Group | Type | nDCG@10 drop |
|---|---|---|---|
| 1 | Candidate–history category match | engineered | 0.0456 |
| 2 | Text vector | learned | 0.0329 |
| 3 | Candidate–history entity match | engineered | 0.0260 |
| 4 | User history (whole) | learned | 0.0196 |
| 5 | Category embeddings | learned | 0.0169 |
| 6 | Article entity statistics | engineered | 0.0084 |
| 7 | Entity vector | learned | 0.0050 |
| 8 | History profile | engineered | 0.0035 |
| 9 | Article text & category flags | engineered | 0.0014 |
| 10 | Impression context | engineered | 0.0010 |
| 11 | Time × category | engineered | 0.0000 |

The two **matching features**, category match and entity match, matter most. They measure directly how close a candidate is to what the user clicked before.

**Feature sets** (starting hyperparameters):

| Feature set | Engineered features | Best epoch | Validation AUC | Validation nDCG@10 |
|---|---|---|---|---|
| **All features (selected)** | 90 | 3 | 0.6566 | **0.3921** |
| Drop time × category | 60 | 3 | 0.6543 | 0.3886 |
| No engineered features | 0 | 10 | 0.6439 | 0.3787 |

- **Engineered features help:** removing all of them costs 0.013 nDCG@10.
- **Time × category:** its permutation importance is zero, yet dropping it lowered nDCG@10 by 0.0035. With a single seed, this is most likely run-to-run noise. Keeping all features is the safe choice.

## Step 7: hyperparameter tuning

Six settings were tried. Their validation nDCG@10 ranged from **0.3849 to 0.3921**.
- The **starting setting was best** (0.3921).
- The next best were dim 64 / MLP [64] / dropout 0.3 (0.3893) and dim 128 / MLP [64] / dropout 0.3 / weight decay 0.0001 (0.3887).
- With dropout 0.3, the best epoch came later (9–11), but the result was no better.
- The dim 64 variants, with 40–62k parameters instead of 103k, were 0.003–0.007 lower.

The model is **not very sensitive** to these settings, so the starting setting was kept.

## Step 9: test result and interpretation

- **Generalisation.** Test AUC (0.6561) matches validation AUC (0.6566).
- **Test nDCG@10 is higher than validation** (0.4051 vs 0.3921) mainly because test slates are easier for any ranker: the random floor is 0.2865 on test vs 0.2703 on validation. Measured against the floor, the gain is the same: +0.119 on test vs +0.122 on validation. There is no sign that the choices over-fitted validation.
- **Cohorts on test.** 88% of test impressions come from users unseen in train + validation, because test comes from a different MIND source (`MINDsmall_dev`). Cold and warm users still score the same (0.4051 vs 0.4050). Empty histories remain the weakest group (0.3377).
- **In-sample rows.** The final model's train and validation rows (nDCG@10 0.5045 and 0.4544) are **in-sample** scores and are not generalisation results.

## Limitations

- **Single seed (42).** Differences below about 0.004 nDCG@10 between runs (Steps 6–7) are within likely seed-to-seed noise.
- **Text encoder.** The frozen text encoder is TF-IDF + SVD, which explains 13.3% of the TF-IDF variance.
- **Batch size.** A batch is 128 sampled groups, or 640 user–candidate pairs, the same as `Candidate_Attention_MLP`.
- **Historical CTR features.** They were left out by design, so popularity signals are not used.